In [1]:
import gym
import math
import random
import statistics
from collections import namedtuple, deque
from itertools import count
import matplotlib
import matplotlib.pyplot as plt
import tqdm
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

In [2]:
env = gym.make('Acrobot-v1')

# configurar matplotlib
is_ipython = 'inline' in matplotlib.get_backend()
if is_ipython:
    from IPython import display

plt.ion()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.10/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


In [3]:
# Se a GPU for usada
device = torch.device(
    "cuda" if torch.cuda.is_available() else
    "mps" if torch.backends.mps.is_available() else
    "cpu"
)

In [4]:
Transition = namedtuple('Transition',
                        ('state', 'action', 'next_state', 'reward'))

In [5]:
class ReplayMemory(object):
    """Classe para armazenar e amostrar transições de memória de repetição"""

    def __init__(self, capacity):
        # Inicializa uma deque (fila de duas extremidades) com um tamanho máximo de `capacity`
        self.memory = deque([], maxlen=capacity)

    def push(self, *args):
        """Salva uma transição"""
        # Adiciona uma nova transição à memória
        self.memory.append(Transition(*args))

    def sample(self, batch_size):
        """Amostra um lote de transições aleatoriamente"""
        # Retorna um número especificado (batch_size) de transições aleatórias da memória
        return random.sample(self.memory, batch_size)

    def __len__(self):
        """Retorna o tamanho atual da memória"""
        # Retorna o número de transições atualmente armazenadas na memória
        return len(self.memory)

In [6]:
class DQN(nn.Module):
    """Rede Neural Profunda para a DQN (Deep Q-Network)"""

    def __init__(self, n_observations, n_actions):
        super(DQN, self).__init__()
        # Primeira camada totalmente conectada (input para hidden)
        self.layer1 = nn.Linear(n_observations, 128)
        # Segunda camada totalmente conectada (hidden para hidden)
        self.layer2 = nn.Linear(128, 128)
        # Terceira camada totalmente conectada (hidden para output)
        self.layer3 = nn.Linear(128, n_actions)

    # Chamado com um elemento para determinar a próxima ação, ou um lote durante a otimização
    # Retorna tensor([[left0exp,right0exp]...]).
    def forward(self, x):
        # Aplicar a função de ativação ReLU após a primeira camada
        x = F.relu(self.layer1(x))
        # Aplicar a função de ativação ReLU após a segunda camada
        x = F.relu(self.layer2(x))
        # Passar pela terceira camada sem ativação para obter os valores Q para cada ação
        return self.layer3(x)

In [7]:
1e-4

0.0001

In [8]:
BATCH_SIZE = 128 # BATCH_SIZE é o número de transições amostradas do buffer de reprodução
GAMMA = 0.99     # GAMMA é o fator de desconto
EPS_START = 0.9  # EPS_START é o valor inicial do épsilon
EPS_END = 0.05   # EPS_END é o valor final do épsilon
EPS_DECAY = 1000 # EPS_DECAY controla a taxa de decaimento exponencial do épsilon, maior significa um decaimento mais lento
TAU = 0.005      # TAU é a taxa de atualização da rede alvo
LR = 1e-4        # LR é a taxa de aprendizado do otimizador ``AdamW``

In [9]:
n_actions = env.action_space.n # Obtenha o número de ações do espaço de ação da academia
state = env.reset() # Obtenha o número de observações de estado
n_observations = len(state)

In [10]:
# Cria a rede neural de política e move para o dispositivo especificado (CPU ou GPU)
policy_net = DQN(n_observations, n_actions).to(device)

# Cria a rede neural alvo e move para o dispositivo especificado
target_net = DQN(n_observations, n_actions).to(device)

# Inicializa a rede neural alvo com os mesmos pesos da rede de política
target_net.load_state_dict(policy_net.state_dict())

<All keys matched successfully>

In [11]:
BUFFER_SIZE = int(1e5)  # Tamanho do buffer de replay
memory = ReplayMemory(BUFFER_SIZE) # Inicializa a memória de replay com a capacidade especificada

# Inicializa o otimizador AdamW com os parâmetros da policy_net, taxa de aprendizado LR, e amsgrad habilitado
# amsgrad=True: Habilita a variante AMSGrad do Adam, que pode proporcionar maior estabilidade em alguns casos
optimizer = optim.AdamW(policy_net.parameters(), lr=LR, amsgrad=True)

In [12]:
steps_done = 0  # Inicializa o contador de passos

def select_action(state):
    global steps_done  # Usa a variável global steps_done para manter o controle do número de passos realizados
    sample = random.random()  # Gera um número aleatório entre 0 e 1
    # Calcula o valor de epsilon (probabilidade de escolher uma ação aleatória)
    eps_threshold = EPS_END + (EPS_START - EPS_END) * \
        math.exp(-1. * steps_done / EPS_DECAY)
    steps_done += 1  # Incrementa o contador de passos

    # Verifica se o número aleatório é maior que o epsilon
    if sample > eps_threshold:
        # Se o número aleatório for maior que o epsilon, selecione a ação com maior recompensa esperada (ação "greedy")
        with torch.no_grad():
            # policy_net(state) retorna um tensor de valores Q para cada ação
            # t.max(1) retorna o valor máximo em cada linha, max(1) retorna (valores, indices)
            # .indices.view(1, 1) retorna o índice da ação com maior valor Q
            return policy_net(state).max(1).indices.view(1, 1)
    else:
        # Caso contrário, selecione uma ação aleatória
        return torch.tensor([[env.action_space.sample()]], device=device, dtype=torch.long)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [13]:
episode_durations = []

def plot_durations(show_result=False):
    """
    Plot the durations of the episodes during training.

    Args:
    show_result (bool, optional): Set True to display the final result plot. Defaults to False.
    """

    # Create a new figure or clear an existing figure
    plt.figure(1)

    # Convert episode_durations list to a tensor of float dtype
    durations_t = torch.tensor(episode_durations, dtype=torch.float)

    if show_result:
        plt.title('Result')
    else:
        plt.clf()
        plt.title('Training...')

    # Set the x and y axis labels
    plt.xlabel('Episode')
    plt.ylabel('Duration')

    # Plot the episode durations
    plt.plot(durations_t.numpy())

    # Take 100 episode averages and plot them too
    if len(durations_t) >= 100:
        means = durations_t.unfold(0, 100, 1).mean(1).view(-1)
        means = torch.cat((torch.zeros(99), means))
        plt.plot(means.numpy())

    # Pause for a bit so that plots are updated
    plt.pause(0.1)

    # If using ipython notebook, clear the output display
    if is_ipython:
        if not show_result:
            display.display(plt.gcf())
            display.clear_output(wait=True)
        else:
            display.display(plt.gcf())

In [14]:
def optimize_model():
    # Verifica se há transições suficientes no replay memory para formar um batch
    if len(memory) < BATCH_SIZE:
        return  # Se não houver, retorna sem fazer nada

    # Amostra um batch de transições do replay memory
    transitions = memory.sample(BATCH_SIZE)

    # Transpõe o batch para obter um Transition de batch-arrays em vez de um batch-array de Transitions
    # Lembre-se que Transition tem forma: Transition = namedtuple('Transition', ('state', 'action', 'next_state', 'reward'))
    # Isso facilita o trabalho com os dados em PyTorch
    batch = Transition(*zip(*transitions))

    # Cria uma máscara de estados não finais e concatena os elementos do batch
    # Um estado final é aquele em que a simulação terminou
    non_final_mask = torch.tensor(tuple(map(lambda s: s is not None,
                                            batch.next_state)), device=device, dtype=torch.bool)

    non_final_next_states = torch.cat([s for s in batch.next_state if s is not None])
    state_batch = torch.cat(batch.state)
    action_batch = torch.cat(batch.action)
    reward_batch = torch.cat(batch.reward)

    # Computa Q(s_t, a) - o modelo computa Q(s_t), então selecionamos as colunas das ações tomadas
    # Estas são as ações que teriam sido tomadas para cada estado do batch de acordo com policy_net
    state_action_values = policy_net(state_batch).gather(1, action_batch)

    # Computa os retornos para todos os próximos estados.
    # Valores esperados das ações para non_final_next_states são computados com  base no target_net "mais antigo"
    # Selecionando a melhor recompensa com max(1).values
    # Isso é mesclado com base na máscara, de modo que teremos o valor do estado esperado ou 0 caso o estado fosse final
    next_state_values = torch.zeros(BATCH_SIZE, device=device)
    with torch.no_grad():
        next_state_values[non_final_mask] = target_net(non_final_next_states).max(1).values

    # Computa os valores Q esperados
    expected_state_action_values = (next_state_values * GAMMA) + reward_batch

    # Computa a perda Huber
    criterion = nn.SmoothL1Loss()
    loss = criterion(state_action_values, expected_state_action_values.unsqueeze(1))

    ################# Otimiza o modelo #################
    optimizer.zero_grad()  # Zera os gradientes acumulados
    loss.backward()        # Computa os gradientes
    # Clipping dos gradientes para evitar explosão de gradientes
    torch.nn.utils.clip_grad_value_(policy_net.parameters(), 100)
    optimizer.step()       # Atualiza os pesos do modelo

In [ ]:
%%time
# Define o número de episódios
num_episodes = 500
# Initialize lists to store average rewards and optimal action percentages
average_rewards = []
optimal_action_percentages = []
for i_episode in range(num_episodes):
    # Initialize the environment and get its state
    state = env.reset()
    state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)

    optimal_action_count = 0
    total_reward = 0

    for t in count():
        # Choose the next action using the current policy network
        action = select_action(state)

        # Check if the selected action is optimal
        with torch.no_grad():
            optimal_action = policy_net(state).max(1)[1].view(1, 1)
        if action == optimal_action:
            optimal_action_count += 1

        # Take the action and observe the next state, reward, and termination signal
        observation, reward, terminated, _ = env.step(action.item())
        total_reward += reward
        reward = torch.tensor([reward], device=device)
        done = terminated

        # Set the next state to None if the game is over, otherwise convert the observation to a tensor and unsqueeze it
        if terminated:
            next_state = None
        else:
            next_state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

        # Store the transition in memory
        memory.push(state, action, next_state, reward)

        # Move to the next state
        state = next_state

        # Perform one step of the optimization on the policy network
        optimize_model()

        # Soft update of the target network's weights
        # θ′ ← τ θ + (1 −τ )θ′
        target_net_state_dict = target_net.state_dict()
        policy_net_state_dict = policy_net.state_dict()
        for key in policy_net_state_dict:
            target_net_state_dict[key] = policy_net_state_dict[key] * TAU + target_net_state_dict[key] * (1 - TAU)
        target_net.load_state_dict(target_net_state_dict)

        # Check if the game is over
        if done:
            # Add the episode duration to the list of durations and plot them
            episode_durations.append(t + 1)
            print(episode_durations[-1])
            plot_durations()

            # End the episode
            break

    # Calculate the average reward for this episode
    average_reward = total_reward / (t + 1)
    average_rewards.append(average_reward)

    # Calculate the optimal action percentage for this episode
    optimal_action_percentage = (optimal_action_count / (t + 1)) * 100
    optimal_action_percentages.append(optimal_action_percentage)

    print(f"Episode {i_episode + 1}/{num_episodes}: Average Reward = {average_reward:.2f}, Optimal Action Percentage = {optimal_action_percentage:.2f}%")
    if optimal_action_percentage >= 99:
        break

# End of the training
print('Complete')
plot_durations(show_result=True)
plt.ioff()
plt.show()

In [ ]:
def plot_metrics():
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.set_title("Average Reward")
    ax1.set_xlabel("Episode")
    ax1.set_ylabel("Value")
    ax1.plot(average_rewards, label="Average Reward")
    ax1.legend(loc="upper left")

    ax2.set_title("% Optimal Action")
    ax2.set_xlabel("Episode")
    ax2.set_ylabel("Value")
    ax2.plot(optimal_action_percentages, label="% Optimal Action")
    ax2.legend(loc="upper left")

    plt.tight_layout()
    plt.show()

# Call the function to plot the metrics
plot_metrics()

In [ ]:
# Initialize the environment
env = gym.make("Acrobot-v1")

# Teste o agente treinado e renderize o ambiente
state = env.reset()
state = torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0)
done = False

# Cria uma figura e um objeto de eixo
fig, ax = plt.subplots()

while not done:
    # Renderiza o ambiente
    ax.imshow(env.render(mode='rgb_array'))
    plt.pause(0.01)

    with torch.no_grad():
        action = policy_net(state).max(1)[1].view(1, 1)

    # Step the environment
    observation, reward, done, _ = env.step(action.item())
    state = torch.tensor(observation, dtype=torch.float32, device=device).unsqueeze(0)

# Close the environment
env.close()